### Initialisation

Doing all the relevant imports

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

# install dependencies
!pip install sympy==1.13.1
!pip install grad-cam --quiet

#do required imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec  # used for showing all images in a grid
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.utils.class_weight import compute_class_weight

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from scipy.optimize import linear_sum_assignment  # for cosine/dsc metrics
from skimage.transform import resize as sk_resize
import shutil  #??
import cv2
import tqdm

# verify gpu
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/ham10000_dataset/dataset/ham10000'
DATA_DIR  = '/content/dataset/ham10000'
REPO_DIR  = '/content/InvertibleCE'
HAM_OUTPUT_DIR = '/content/drive/MyDrive/ham10000_dataset/outputs'

os.makedirs(HAM_OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

DRIVE_ROOT    = '/content/drive/MyDrive/derm7pt'
DERM7PT_ZIP   = f'{DRIVE_ROOT}/release_v0.zip'
OUTPUT_DIR    =f'{DRIVE_ROOT}/outputs'
LOCAL_DATA  = '/content/derm7pt'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)

## Handling HAM10000 and Derm7pt data and repos

This code takes the HAM10000 dataset, Derm7pt dataset, and ICE repo and saves them to drive if not already done. Otherwise it simply loads them.

In [ ]:
if not os.path.exists(REPO_DIR):
    print('Cloning repo and installing dependencies...')
    !git clone https://github.com/zhangrh93/InvertibleCE {REPO_DIR}
    os.chdir(REPO_DIR)
    !sed -i 's/torch==.*/torch/' requirements.txt
    !sed -i 's/torchvision==.*/torchvision/' requirements.txt
    !pip install -r requirements.txt
    print('Done.')
else:
    print('Repo already cloned, skipping.')
    os.chdir(REPO_DIR)

EXPLAINERS_DRIVE_DIR = os.path.join(HAM_OUTPUT_DIR, 'Explainers')
os.makedirs(EXPLAINERS_DRIVE_DIR, exist_ok=True)

local_explainers_path = os.path.join(REPO_DIR, 'Explainers')
if os.path.islink(local_explainers_path):
    print('Explainers symlink already exists, skipping.')
elif os.path.exists(local_explainers_path):
    for item in os.listdir(local_explainers_path):
      src = os.path.join(local_explainers_path, item)
      dst = os.path.join(EXPLAINERS_DRIVE_DIR, item)
      if os.path.isdir(src):
          shutil.copytree(src, dst, dirs_exist_ok=True)
          shutil.rmtree(src)
      else:
          shutil.copy2(src, dst)
          os.remove(src)
    os.rmdir(local_explainers_path)
    os.symlink(EXPLAINERS_DRIVE_DIR, local_explainers_path)
    print('Migrated existing local explainers to Drive and symlinked.')
else:
    os.symlink(EXPLAINERS_DRIVE_DIR, local_explainers_path)
    print('Created Explainers -> Drive symlink.')

# Ensure HAM10000_metadata.csv is present and not empty
HAM_METADATA_PATH = f'{DATA_DIR}/HAM10000_metadata.csv'
if not os.path.exists(HAM_METADATA_PATH) or os.path.getsize(HAM_METADATA_PATH) == 0:
    print('Copying dataset from Drive...')
    # Ensure DATA_DIR is clean of previous HAM10000 data before copying
    !rm -f {DATA_DIR}/*.jpg
    !rm -f {HAM_METADATA_PATH}
    !cp {DRIVE_DIR}/*.jpg {DATA_DIR}/
    !cp {DRIVE_DIR}/HAM10000_metadata.csv {DATA_DIR}/
    print('Done.')
else:
    print('Already copied, skipping.')

meta = pd.read_csv(f'{DATA_DIR}/HAM10000_metadata.csv')
print(f"Total rows: {len(meta)}")
print(meta['dx'].value_counts())

import ICE.Data
import ICE.ModelWrapper
import ICE.Explainer
import ICE.utils

HAM_CLASSES = {
    'mel': 0,
    'nv': 1,
    'bcc': 2,
    'akiec': 3,
    'bkl': 4,
    'df': 5,
    'vasc': 6
}

meta = pd.read_csv(f'{DATA_DIR}/HAM10000_metadata.csv')
img_dir = Path(DATA_DIR)

# map each class to its corresponding images (and indices)
ham_dict = {}
for label_name, label_idx in HAM_CLASSES.items():
    paths = meta[meta['dx'] == label_name]['image_id'].apply(
        lambda x: str(img_dir / f"{x}.jpg")
    ).tolist()
    ham_dict[label_idx] = {
        'idx': label_idx,
        'name': label_name,
        'paths': paths
    }
for d in list(ham_dict.values()):
    print(d)

# define parameters
paras = {}
paras['target_classes'] = [HAM_CLASSES['mel']]
paras['classes_names'] = ['melanoma']
paras['n_components'] = 10
paras['layer_name'] = 'layer4'
paras['title'] = "ham_melanoma"
paras['overwrite'] = False

In [ ]:
import zipfile

# Check if it's already extracted correctly before doing anything else
matches = list(Path(LOCAL_DATA).rglob('meta.csv'))

if matches:
    print(f'Found existing extraction, skipping unzip.')
else:
    print('Not found, wiping and unzipping fresh...')
    shutil.rmtree(LOCAL_DATA, ignore_errors=True)
    os.makedirs(LOCAL_DATA, exist_ok=True)
    with zipfile.ZipFile(DERM7PT_ZIP, 'r') as zf:
        zf.extractall(LOCAL_DATA)
    print('Done.')
    matches = list(Path(LOCAL_DATA).rglob('meta.csv'))
    if not matches:
        raise FileNotFoundError('Could not find meta.csv even after unzipping, check DERM7PT_ZIP contents.')

LOCAL_DATA = str(matches[0].parent.parent)
print(f'Dataset root: {LOCAL_DATA}')

for required in ['meta/meta.csv', 'meta/train_indexes.csv',
                 'meta/valid_indexes.csv', 'meta/test_indexes.csv']:
    path = os.path.join(LOCAL_DATA, required)
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {status}  {required}')

CONFIG = {
    'dataset_root': LOCAL_DATA,
    'output_dir':   OUTPUT_DIR,
    'concepts': {
        'pigment_network':      {'labels': ['absent', 'typical', 'atypical'],  'n_classes': 3},
        'blue_whitish_veil':    {'labels': ['absent', 'present'],              'n_classes': 2},
        'vascular_structures':  {'labels': ['absent', 'regular', 'irregular'], 'n_classes': 3, 'label_mapping': {'arborizing': 'regular', 'comma': 'regular', 'hairpin': 'regular', 'within regression': 'regular', 'wreath': 'regular', 'dotted': 'irregular', 'linear irregular': 'irregular'}},
        'pigmentation':         {'labels': ['absent', 'regular', 'irregular'], 'n_classes': 3, 'label_mapping': {'diffuse regular': 'regular', 'localized regular': 'regular', 'diffuse irregular': 'irregular', 'localized irregular': 'irregular'}},
        'streaks':              {'labels': ['absent', 'regular', 'irregular'], 'n_classes': 3},
        'dots_and_globules':        {'labels': ['absent', 'regular', 'irregular'], 'n_classes': 3},
        'regression_structures':{'labels': ['absent', 'present'],              'n_classes': 2, 'label_mapping': {'blue areas': 'present', 'white areas': 'present', 'combinations': 'present',}},
    },
    'image_size':    224,
    'batch_size':    16,
    'num_epochs':    30,
    'learning_rate': 1e-4,
    'weight_decay':  1e-4,
    # Change this to 'gradcam', 'scorecam', or 'eigencam' to compare methods
    'cam_method':   'gradcam++',
    'device':        device,
}

print('Configuration set.')
print(f"  Dataset : {CONFIG['dataset_root']}")
print(f"  Outputs : {CONFIG['output_dir']}")
print(f"  CAM     : {CONFIG['cam_method']}")

In [ ]:
class Derm7ptDataset(Dataset):

    def __init__(self, dataset_root, split_indexes, concepts_config, transform=None):
        self.root = Path(dataset_root)
        self.transform = transform
        self.concepts_config = concepts_config
        df = pd.read_csv(self.root / 'meta' / 'meta.csv')
        self.df = df.iloc[split_indexes].reset_index(drop=True)
        self.label_encoders = {
            concept: {label: idx for idx, label in enumerate(cfg['labels'])}
            for concept, cfg in concepts_config.items()
        }
        self.precompute_numeric_labels()  # combining extra labels into known ones

        # Add binary diagnosis labels for Melanoma (0) vs Nevus (1)
        self.diagnosis_map = {
            'melanoma': 0, 'melanoma (in situ)': 0, 'melanoma (less than 0.76 mm)': 0,
            'melanoma (0.76 to 1.5 mm)': 0, 'melanoma (more than 1.5 mm)': 0, 'melanoma metastasis': 0,
            'clark nevus': 1, 'reed or spitz nevus': 1, 'blue nevus': 1,
            'combined nevus': 1, 'congenital nevus': 1, 'dermal nevus': 1, 'recurrent nevus': 1,
        }
        self.df['binary_diagnosis_label'] = self.df['diagnosis'].astype(str).str.lower().str.strip().map(self.diagnosis_map).fillna(-1).astype(int)

    def precompute_numeric_labels(self):
        for concept, cfg in self.concepts_config.items():
            mapping = cfg.get('label_mapping', {})
            raw = self.df[concept].astype(str).str.lower().str.strip()
            raw = raw.map(lambda x: mapping.get(x, x))
            self.df[f'{concept}_numeric'] = raw.map(self.label_encoders[concept]).fillna(-1).astype(int)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root / 'images' / row['derm']
        image = Image.open(img_path).convert('RGB')
        w, h = image.size  # these 3 lines were added to crop the border, this is present in the derm7pt code
        crop_amount = 25
        image = image.crop((crop_amount, crop_amount, w - crop_amount, h - crop_amount))
        if self.transform:
            image = self.transform(image)
        labels = {concept: int(row[f'{concept}_numeric']) for concept in self.concepts_config}
        diagnosis_label = int(row['binary_diagnosis_label'])
        return image, labels, row['derm'], diagnosis_label


def load_split_indexes(dataset_root, split_name):
    path = Path(dataset_root) / 'meta' / f'{split_name}_indexes.csv'
    return list(pd.read_csv(path)['indexes'])


def get_transforms(image_size, split):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((image_size + 32, image_size + 32)),
            transforms.RandomCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    concept_labels = [b[1] for b in batch]
    filenames = [b[2] for b in batch]
    diagnosis_labels = torch.tensor([b[3] for b in batch], dtype=torch.long)
    return images, concept_labels, filenames, diagnosis_labels


# Load splits
cfg = CONFIG
train_idx = load_split_indexes(cfg['dataset_root'], 'train')
valid_idx = load_split_indexes(cfg['dataset_root'], 'valid')
test_idx  = load_split_indexes(cfg['dataset_root'], 'test')

train_dataset = Derm7ptDataset(cfg['dataset_root'], train_idx, cfg['concepts'],
                                transform=get_transforms(cfg['image_size'], 'train'))
val_dataset   = Derm7ptDataset(cfg['dataset_root'], valid_idx, cfg['concepts'],
                                transform=get_transforms(cfg['image_size'], 'valid'))
test_dataset  = Derm7ptDataset(cfg['dataset_root'], test_idx,  cfg['concepts'],
                                transform=get_transforms(cfg['image_size'], 'test'))
for concept, ccfg in cfg['concepts'].items():
    mapping = ccfg.get('label_mapping', {})
    raw_vals = set(train_dataset.df[concept].astype(str).str.lower().str.strip().unique())
    mapped_vals = {mapping.get(v, v) for v in raw_vals}
    known = set(cfg['concepts'][concept]['labels'])
    unknown = mapped_vals - known
    print(f"{concept:25s} known={known}  unmapped={unknown}")

derm_train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'],
                          shuffle=True,  num_workers=2, pin_memory=True, collate_fn=collate_fn)
derm_val_loader   = DataLoader(val_dataset,   batch_size=cfg['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
derm_test_loader = DataLoader(test_dataset, batch_size=cfg['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
class HAM10000BinaryDataset(Dataset):
    def __init__(self, ham_dict, melanoma_idx, nevus_idx, transform=None):
        self.transform = transform
        self.data = []
        # Collect melanoma images
        for path in ham_dict[melanoma_idx]['paths']:
            self.data.append((path, 0)) # 0 for melanoma
        # Collect nevus images
        for path in ham_dict[nevus_idx]['paths']:
            self.data.append((path, 1)) # 1 for nevus

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

melanoma_idx = HAM_CLASSES['mel']
nevus_idx = HAM_CLASSES['nv']

ham_transform = transforms.Compose([
    transforms.Resize((cfg['image_size'], cfg['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ham_binary_dataset = HAM10000BinaryDataset(ham_dict, melanoma_idx, nevus_idx, ham_transform)

from sklearn.model_selection import train_test_split

indices = list(range(len(ham_binary_dataset)))
labels_for_split = [item[1] for item in ham_binary_dataset.data]

train_val_indices, test_indices, train_val_labels, _ = train_test_split(
    indices, labels_for_split, test_size=0.40, random_state=42, stratify=labels_for_split
)

train_indices, val_indices, _, _ = train_test_split(
    train_val_indices, train_val_labels, test_size=1/3, random_state=42, stratify=train_val_labels
)

ham_train_subset = torch.utils.data.Subset(ham_binary_dataset, train_indices)
ham_val_subset = torch.utils.data.Subset(ham_binary_dataset, val_indices)
ham_test_subset = torch.utils.data.Subset(ham_binary_dataset, test_indices)

ham_loader = DataLoader(ham_binary_dataset, batch_size=cfg['batch_size'], shuffle=False, num_workers=2)

print(f"HAM10000 Binary Dataset (Melanoma/Nevus): {len(ham_binary_dataset)} samples")
print(f"  Train subset: {len(ham_train_subset)} samples")
print(f"  Val subset: {len(ham_val_subset)} samples")
print(f"  Test subset: {len(ham_test_subset)} samples")

### Create ICE explainer functions

These three functions are important to how the ICE explainer works, and will streamline work down the track.

In [ ]:
def make_imgs(label_idx):
  '''
  Makes returns a list of all image paths for the class specified by label_idx.
  '''
  return [(p, label_idx) for p in ham_dict[label_idx]['paths']]

def get_loaders(target_classes = []):
  '''
  Returns a list of data loaders for each class specified by target_classes.

  ICE needs separate loaders for each class to learn class-specific concepts.
  '''
  loaders = []
  for i,idx in enumerate(target_classes):
    loaders.append(DataLoader(
            ICE.Data.ImageDataset(make_imgs(idx),
                    transforms.Compose([
                    transforms.Resize((224,224)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                            std=[0.229, 0.224, 0.225]),
                    ])),
            batch_size = 16,
            num_workers=2,
            #shuffle=True
    ))
  return loaders


def get_explainer(paras, model):
  classes_names = paras['classes_names']
  target_classes = paras['target_classes']
  n_components = paras['n_components'] # number of concepts to generate
  layer_name = paras['layer_name']
  title = paras['title']
  overwrite = paras['overwrite']

  # If overwriting, delete the old folder
  if overwrite:
    try:
        shutil.rmtree('Explainers/'+title)
    except:
        pass

  # Just predict for the specific target classes
  model.set_predict_target(target_classes)
  loaders = get_loaders(target_classes)

  # load the data if it exists already
  if os.path.exists(Path('Explainers')/title):
    Exp = ICE.Explainer.Explainer(title = title)
    Exp.load()
    print ('model loaded')
  else:
    # create an Explainer
    Exp = ICE.Explainer.Explainer(title = title,
                    layer_name = layer_name,
                    class_names = classes_names,
                    utils = ICE.utils.img_utils(mode = "torch"),
                    n_components = n_components,
                    reducer_type = "NMF"
                  )

    loaders = [DataLoader(
      Subset(loader.dataset, range(min(1000, len(loader.dataset)))),
      batch_size=16,
      num_workers=2
    ) for loader in loaders]

    # train reducer based on target classes
    Exp.train_model(model,loaders)
    # generate features
    Exp.generate_features(model, loaders)
    # save the explainer, use load to load it with the same title
    Exp.save()

  return Exp

### Predicting Derm7pt concepts from images

Taking a pretrained ResNet50 model, and giving it the capability to predict all 7 Derm7pt concepts (and computing class weights) - this is the first half of the CBM (images -> concepts)

In [ ]:
class Derm7ptMultiTaskModel(nn.Module):
    def __init__(self, concepts_config, dropout_rate=0.5, concept_channels=256):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4, # <- remove for better resolution
        )
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout_rate)
        self.concept_convs   = nn.ModuleDict({
            concept: nn.Conv2d(2048, concept_channels, kernel_size=1)  # was 1024 (for layer 4 exclusion)
            for concept, cfg in concepts_config.items()
        })

        self.heads = nn.ModuleDict({
            concept: nn.Linear(concept_channels, cfg['n_classes'])
            for concept, cfg in concepts_config.items()
        })

    def forward(self, x):
        feat   = self.features(x)
        outputs = {}
        for concept, conv in self.concept_convs.items():
            c_feat = conv(feat)
            pooled = self.dropout(self.pool(c_feat).flatten(1))
            outputs[concept] = self.heads[concept](pooled)
        return outputs

def compute_concept_class_weights(dataset, concepts_config, device):
    all_labels = {c: [] for c in concepts_config}
    for _, label_dict, _, _ in dataset:
        for concept, label_idx in label_dict.items():
            if label_idx >= 0:
                all_labels[concept].append(label_idx)
    weights = {}
    for concept, cfg in concepts_config.items():
        labels_arr = np.array(all_labels[concept])
        present_classes    = np.unique(labels_arr)  # some small classes might not be in training set
        cw_present = compute_class_weight('balanced', classes=present_classes, y=labels_arr)

        mean_weight = cw_present.mean()
        cw = np.full(cfg['n_classes'], mean_weight, dtype=np.float64)
        for cls, w in zip(present_classes, cw_present):
            cw[cls] = w

        weights[concept] = torch.tensor(cw, dtype=torch.float).to(device)
        print(f'  {concept:30s}: {cw.round(2)}')
    return weights

derm_model = Derm7ptMultiTaskModel(cfg['concepts']).to(device)
print('Model built: ResNet-50 backbone + 7 concept heads')
total_params = sum(p.numel() for p in derm_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

print('Computing class weights...')
derm_class_weights = compute_concept_class_weights(train_dataset, cfg['concepts'], device)

### Training HAM10000 Model and ICE Explainer

In [ ]:
NUM_CLASSES = 7

m = models.resnet50(pretrained=True)
m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
m = m.cuda()

# Getting a map of labels and classes over the training subset
train_indices_set = set(train_indices)
all_imgs = []
all_labels = []
for idx in HAM_CLASSES.values():
    imgs = make_imgs(idx)

# Create training image dataset using the training subset
train_imgs = [ham_binary_dataset.data[i] for i in train_indices]
train_labels = [label for _, label in train_imgs]

# Weighted bias based on training class distribution
class_counts = [train_labels.count(i) for i in range(2)]
ham_class_weights = torch.tensor([1.0 / max(c, 1) for c in class_counts], dtype=torch.float).cuda()
ham_criterion = nn.CrossEntropyLoss(weight=ham_class_weights)

ham_optimizer = optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-4)

train_loader = DataLoader(
    ham_train_subset,
    batch_size=16,
    num_workers=2,
    shuffle=True
)

val_loader = DataLoader(
    ham_val_subset,
    batch_size=16,
    num_workers=2,
    shuffle=False
)

NUM_EPOCHS = 20
scheduler = optim.lr_scheduler.CosineAnnealingLR(ham_optimizer, T_max=NUM_EPOCHS)
HAM_CHECKPOINT = os.path.join(OUTPUT_DIR, 'ham_resnet50.pth')

if os.path.exists(HAM_CHECKPOINT):
    m.load_state_dict(torch.load(HAM_CHECKPOINT, map_location=device))
    print('Loaded existing HAM10000 classifier.')
else:
    m.train()
    for epoch in range(NUM_EPOCHS):
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.permute(0, 3, 1, 2).cuda(), labels.cuda()
            ham_optimizer.zero_grad()
            outputs = m(inputs)

            restricted_outputs = outputs[:, [HAM_CLASSES['mel'], HAM_CLASSES['nv']]]
            loss = ham_criterion(restricted_outputs, labels)
            loss.backward()
            ham_optimizer.step()
            running_loss += loss.item()
            _, predicted = restricted_outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        scheduler.step()

        m.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for val_inputs, val_labels in val_loader:
                val_inputs, val_labels = val_inputs.permute(0, 3, 1, 2).cuda(), val_labels.cuda()
                val_outputs = m(val_inputs)
                val_restricted = val_outputs[:, [HAM_CLASSES['mel'], HAM_CLASSES['nv']]]
                _, val_predicted = val_restricted.max(1)
                val_total += val_labels.size(0)
                val_correct += val_predicted.eq(val_labels).sum().item()
        m.train()

        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {running_loss/len(train_loader):.4f}, Train Acc: {100.*correct/total:.1f}%, Val Acc: {100.*val_correct/val_total:.1f}%")

    torch.save(m.state_dict(), HAM_CHECKPOINT)

m.eval()
ham_model = ICE.ModelWrapper.PytorchModelWrapper(
    m,
    batch_size=16,
    predict_target=paras['target_classes'],
    input_channel_first=False,
    model_channel_first=True
)

In [ ]:
# Two-class explainer: melanoma vs nevi
target_classes = [HAM_CLASSES['mel'], HAM_CLASSES['nv']]
classes_names = ['melanoma', 'nevi']
title = "ham_mel_vs_nv"
n_components = 10
layer_name = 'layer3'
overwrite = True

paras = {}
paras['target_classes'] = target_classes
paras['classes_names'] = classes_names
paras['n_components'] = n_components
paras['layer_name'] = layer_name
paras['title'] = title
paras['overwrite'] = overwrite

Exp = get_explainer(paras, ham_model)

fidelity = Exp.reducer_err
for name, err in zip(classes_names, fidelity):
    print(f"Fidelity error ({name}): {err:.4f} ({err*100:.2f}%)")

### Saving the ICE concept images to use later as a reference

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

mel_idx = HAM_CLASSES['mel']
nv_idx  = HAM_CLASSES['nv']

@torch.no_grad()
def collect_binary_mel_nv_preds(model, loader, mel_idx, nv_idx, device):
    model.eval()
    all_preds = []
    all_labels = []
    for inputs, labels in loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        # restrict to just the two target classes, matching set_predict_target's framing
        restricted = outputs[:, [mel_idx, nv_idx]]
        binary_pred = restricted.argmax(dim=1)  # 0 -> mel, 1 -> nv
        all_preds.extend(binary_pred.cpu().numpy().tolist())
        all_labels.extend(labels.numpy().tolist())
    return np.array(all_preds), np.array(all_labels)

# Create test loader for held-out evaluation
test_loader = DataLoader(ham_test_subset, batch_size=16, shuffle=False, num_workers=2)

preds, labels = collect_binary_mel_nv_preds(m, test_loader, mel_idx, nv_idx, device)

# only keep rows that are actually mel or nv
mask = np.isin(labels, [0, 1])
binary_true = labels[mask]
binary_pred = preds[mask]

acc = accuracy_score(binary_true, binary_pred)
f1  = f1_score(binary_true, binary_pred, average='macro', zero_division=0)
print(f"Restricted mel-vs-nv (explainer framing) on HAM10000 Test Set — Accuracy: {acc*100:.2f}%  F1: {f1:.3f}")

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

# Calculate per-class accuracy

# Melanoma (where binary_true is 0)
mel_mask = (binary_true == 0)
mel_true = binary_true[mel_mask]
mel_pred = binary_pred[mel_mask]
mel_acc = accuracy_score(mel_true, mel_pred)
mel_f1 = f1_score(mel_true, mel_pred, average='binary', pos_label=0, zero_division=0)

# Nevus (where binary_true is 1)
nv_mask = (binary_true == 1)
nv_true = binary_true[nv_mask]
nv_pred = binary_pred[nv_mask]
nv_acc = accuracy_score(nv_true, nv_pred)
nv_f1 = f1_score(nv_true, nv_pred, average='binary', pos_label=1, zero_division=0)

print(f"Melanoma (explainer framing) Accuracy: {mel_acc*100:.2f}%  F1: {mel_f1:.3f}")
print(f"Nevus (explainer framing) Accuracy: {nv_acc*100:.2f}%  F1: {nv_f1:.3f}")

### Training baseline Derm7pt model

(dataset -> diagnosis, no concept bottleneck)

In [ ]:
# --- Filter to only melanoma/nevus rows (drop -1 labels) ---
def make_binary_subset(dataset):
    valid_idx = [i for i in range(len(dataset)) if dataset.df.loc[i, 'binary_diagnosis_label'] != -1]
    return Subset(dataset, valid_idx)

binary_train_subset = make_binary_subset(train_dataset)
binary_val_subset   = make_binary_subset(val_dataset)
binary_test_subset  = make_binary_subset(test_dataset)

binary_train_loader = DataLoader(binary_train_subset, batch_size=cfg['batch_size'],
                                  shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
binary_val_loader   = DataLoader(binary_val_subset,   batch_size=cfg['batch_size'],
                                  shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
binary_test_loader  = DataLoader(binary_test_subset,  batch_size=cfg['batch_size'],
                                  shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

print(f'Binary train: {len(binary_train_subset)} | val: {len(binary_val_subset)} | test: {len(binary_test_subset)}')

In [ ]:
NUM_CLASSES_BINARY = 2  # 0 = melanoma, 1 = nevus

m_binary = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
m_binary.fc = nn.Linear(m_binary.fc.in_features, NUM_CLASSES_BINARY)
m_binary = m_binary.cuda()

binary_labels_all = [train_dataset.df.loc[i, 'binary_diagnosis_label'] for i in binary_train_subset.indices]
binary_class_counts = [binary_labels_all.count(i) for i in range(NUM_CLASSES_BINARY)]
binary_class_weights = torch.tensor([1.0 / c for c in binary_class_counts], dtype=torch.float).cuda()

binary_criterion = nn.CrossEntropyLoss(weight=binary_class_weights)
binary_optimizer = optim.Adam(m_binary.parameters(), lr=1e-4)

NUM_EPOCHS = 20
BINARY_CHECKPOINT = os.path.join(OUTPUT_DIR, 'derm7pt_binary_resnet50.pth')

if os.path.exists(BINARY_CHECKPOINT):
    m_binary.load_state_dict(torch.load(BINARY_CHECKPOINT, map_location=device))
    print('Loaded existing Derm7pt binary classifier.')
else:
    m_binary.train()
    for epoch in range(NUM_EPOCHS):
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, _concept_labels, _filenames, labels in binary_train_loader:
            inputs, labels = inputs.permute(0, 3, 1, 2).cuda(), labels.cuda()
            binary_optimizer.zero_grad()
            outputs = m_binary(inputs)
            loss = binary_criterion(outputs, labels)
            loss.backward()
            binary_optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {running_loss/len(binary_train_loader):.4f}, Acc: {100.*correct/total:.1f}%")

    torch.save(m_binary.state_dict(), BINARY_CHECKPOINT)

m_binary.eval()

# Training Derm7pt CBM (first half)
Trains the image -> concept CBM, and applies per-concept GradCAM

In [ ]:
def train_one_epoch(model, loader, optimizer, class_weights, device):
    model.train()
    total_loss = 0.0

    concept_loss_totals = {c: 0.0 for c in class_weights}
    concept_loss_counts = {c: 0 for c in class_weights}

    for images, label_dicts, _, _ in loader:
        images = images.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        concept_losses = {}
        for concept, logits in outputs.items():
            labels = torch.tensor(
                [d[concept] for d in label_dicts], dtype=torch.long
            ).to(device)
            valid = labels >= 0
            if valid.sum() == 0:
                continue
            criterion = nn.CrossEntropyLoss(weight=class_weights[concept], ignore_index=-1)
            concept_losses[concept] = criterion(logits[valid], labels[valid])

        batch_loss = sum(concept_losses.values())
        batch_loss.backward()
        optimizer.step()
        total_loss += batch_loss.item()
        for concept, loss_val in concept_losses.items():
            concept_loss_totals[concept] += loss_val.item()
            concept_loss_counts[concept] += 1

    avg_concept_losses = {
        c: (concept_loss_totals[c] / concept_loss_counts[c] if concept_loss_counts[c] > 0 else float('nan'))
        for c in class_weights
    }
    return total_loss / len(loader), avg_concept_losses


@torch.no_grad()
def evaluate(model, loader, class_weights, device):
    model.eval()
    total_loss = 0.0
    correct = {c: 0 for c in cfg['concepts']}
    counts  = {c: 0 for c in cfg['concepts']}
    for images, label_dicts, _, _ in loader:
        images  = images.to(device)
        outputs = model(images)
        for concept, logits in outputs.items():
            labels = torch.tensor(
                [d[concept] for d in label_dicts], dtype=torch.long
            ).to(device)
            valid = labels >= 0
            if valid.sum() == 0:
                continue
            criterion = nn.CrossEntropyLoss(weight=class_weights[concept], ignore_index=-1)
            total_loss += criterion(logits[valid], labels[valid]).item()
            preds = logits[valid].argmax(dim=1)
            correct[concept] += (preds == labels[valid]).sum().item()
            counts[concept]  += valid.sum().item()
    accs = {c: (correct[c] / counts[c] if counts[c] > 0 else 0.0) for c in correct}
    return total_loss / len(loader), accs

def train_derm_model(cfg, seed, checkpoint_path, device):
    derm_model = Derm7ptMultiTaskModel(cfg['concepts']).to(device)

    if os.path.exists(checkpoint_path):
        print(f'Found existing checkpoint at {checkpoint_path} — loading.')
        derm_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        return derm_model

    derm_optimizer = optim.Adam(derm_model.parameters(),
                           lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(derm_optimizer, T_max=cfg['num_epochs']) # should this be different?
    best_val_loss = float('inf')

    concept_loss_history = {c: [] for c in cfg['concepts']}

    for epoch in range(cfg['num_epochs']):
        train_loss, train_concept_losses = train_one_epoch(derm_model, derm_train_loader, derm_optimizer, derm_class_weights, device)
        val_loss, val_accs = evaluate(derm_model, derm_val_loader, derm_class_weights, device)
        scheduler.step() # what does this do?

        for concept, loss_val in train_concept_losses.items():
            concept_loss_history[concept].append(loss_val)

        print(f'[seed={seed}] Epoch {epoch+1:02d}/{cfg["num_epochs"]} | '
              f'Train: {train_loss:.4f} | Val: {val_loss:.4f}')
        for concept in train_concept_losses:
            print(f'   {concept:30s}: loss={train_concept_losses[concept]:.4f}  acc={val_accs[concept]*100:.1f}%')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(derm_model.state_dict(), checkpoint_path)
            print(f'   ✓ Saved (val_loss={val_loss:.4f}) → {checkpoint_path}')

    plt.figure(figsize=(10, 6))
    for concept, losses in concept_loss_history.items():
        plt.plot(losses, label=concept)
    plt.xlabel('Epoch')
    plt.ylabel('Train Loss')
    plt.legend(fontsize=8)
    plt.title(f'Per-Concept Training Loss (seed={seed})')
    plt.savefig(os.path.join(OUTPUT_DIR, f'concept_loss_curves_seed{seed}.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Training complete for seed={seed}.')
    return derm_model

N_RUNS = 5
SEEDS = [0, 1, 2, 3, 4]

DERM_MODEL_CHECKPOINTS = [
    os.path.join(OUTPUT_DIR, f'best_model_run{i+1}.pth') for i in range(N_RUNS)
]

for seed, checkpoint_path in zip(SEEDS, DERM_MODEL_CHECKPOINTS):
    print(f"\n=== Run {seed+1}/{N_RUNS} (seed={seed}) ===")
    train_derm_model(cfg, seed, checkpoint_path, device)

### Creating second half of the Derm7pt CBM (concepts to diagnosis)

Extracting the predicted concepts from the Derm model and using them as inputs to the second half of the CBM (which is a simple linear classifier that takes concepts and predicts melanoma or nevus from them)

In [ ]:
class DiagnosisClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

# Function to get concepts from derm model
@torch.no_grad()
def extract_predicted_concept_labels_as_features(model, loader, concepts_config, device, ham):
    model.eval()
    all_features = []
    all_diagnosis_labels = []

    for item in loader:
        if ham:
            images, diagnosis_labels = item
        else:
            images, concept_label_dicts, _, diagnosis_labels = item
        images = images.to(device)
        # Filter out invalid diagnosis labels for meta-classifier training/testing
        # 0 for melanoma, 1 for nevus
        valid_diagnosis_mask = (diagnosis_labels == 0) | (diagnosis_labels == 1)
        if not valid_diagnosis_mask.any():
            continue

        valid_images = images[valid_diagnosis_mask]
        valid_diagnosis_labels = diagnosis_labels[valid_diagnosis_mask]

        if valid_images.shape[0] == 0:
            continue

        outputs = model(valid_images)

        # Collect one-hot encoded predicted labels for each concept
        concept_predictions_one_hot = []
        for concept, concept_cfg in concepts_config.items():
            logits = outputs[concept]
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e10, neginf=-1e10)
            predicted_class_indices = logits.argmax(dim=1)
            num_classes_for_concept = concept_cfg['n_classes']
            predicted_class_indices = predicted_class_indices.clamp(min=0, max=num_classes_for_concept - 1)

            one_hot = torch.nn.functional.one_hot(
                predicted_class_indices, num_classes=num_classes_for_concept
            ).float()
            concept_predictions_one_hot.append(one_hot)
        combined_features = torch.cat(concept_predictions_one_hot, dim=1)

        all_features.append(combined_features.cpu())
        all_diagnosis_labels.append(valid_diagnosis_labels.cpu())

    return torch.cat(all_features, dim=0), torch.cat(all_diagnosis_labels, dim=0)

@torch.no_grad()
def evaluate_diagnosis_classifier(model, features, labels, device):
    model.eval()
    features = features.to(device)
    labels = labels.to(device)
    outputs = model(features)
    _, predicted = torch.max(outputs, 1)

    accuracy = accuracy_score(labels.cpu().numpy(), predicted.cpu().numpy())
    f1 = f1_score(labels.cpu().numpy(), predicted.cpu().numpy(), average='macro')

    return accuracy, f1

### Training and evaluating models

Evaluating performance on Derm7pt and HAM10000 datasets

In [ ]:
print("Extracting predicted concept labels (one-hot) as features for training...")
concept_train_features, concept_train_labels = extract_predicted_concept_labels_as_features(derm_model, derm_train_loader, cfg['concepts'], device, False)

print("Extracting predicted concept labels (one-hot) as features for testing...")
og_concept_test_features, og_concept_test_labels = extract_predicted_concept_labels_as_features(derm_model, derm_test_loader, cfg['concepts'], device, False)

# Create a DataLoader for the HAM10000 test subset
ham_test_loader = DataLoader(ham_test_subset, batch_size=cfg['batch_size'], shuffle=False, num_workers=2)
new_concept_test_features, new_concept_test_labels = extract_predicted_concept_labels_as_features(derm_model, ham_test_loader, cfg['concepts'], device, True)

print(f"Shape of training features (from predicted labels): {concept_train_features.shape}")

### Calculating fidelity

Based on baseline models

In [ ]:
torch.manual_seed(0)
clf = DiagnosisClassifier(concept_train_features.shape[1]).to(device)
opt = optim.Adam(clf.parameters(), lr=1e-2)
crit = nn.CrossEntropyLoss()
X, y = concept_train_features.to(device), concept_train_labels.to(device)
for _ in range(500):
    opt.zero_grad()
    crit(clf(X), y).backward()
    opt.step()

og_acc_labels, og_f1   = evaluate_diagnosis_classifier(clf, og_concept_test_features,  og_concept_test_labels,  device)  # Derm7pt
new_acc_labels, new_f1 = evaluate_diagnosis_classifier(clf, new_concept_test_features, new_concept_test_labels, device)  # HAM

In [ ]:
@torch.no_grad()
def get_baseline_diagnosis_accuracy(model, loader, device, ham=False, ham_mel_idx=None, ham_nv_idx=None):
    """Mirrors the valid_diagnosis_mask filtering used in
    extract_predicted_concept_labels_as_features, so the baseline is
    computed over the exact same samples as og/new_concept_test_labels."""
    model.eval()
    correct, total = 0, 0
    for item in loader:
        if ham:
            images, diagnosis_labels = item
        else:
            images, _concept_label_dicts, _fnames, diagnosis_labels = item

        valid_mask = (diagnosis_labels == 0) | (diagnosis_labels == 1)
        if not valid_mask.any():
            continue

        images = images[valid_mask].to(device)
        labels = diagnosis_labels[valid_mask].to(device)

        logits = model(images)
        if ham:
            logits = logits[:, [ham_mel_idx, ham_nv_idx]]  # collapse 7-way -> [mel, nv]

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total if total > 0 else 0.0

derm_baseline_acc = get_baseline_diagnosis_accuracy(m_binary, derm_test_loader, device, ham=False)
ham_baseline_acc = get_baseline_diagnosis_accuracy(
    m, ham_loader, device, ham=True, ham_mel_idx=mel_idx, ham_nv_idx=nv_idx
)

epsilon = 1e-4

derm_fid_err = abs(derm_baseline_acc - new_acc_labels) / (abs(derm_baseline_acc) + epsilon)
ham_fid_err  = abs(derm_baseline_acc - og_acc_labels) / (abs(derm_baseline_acc) + epsilon)

print(f"Derm7pt  - baseline acc: {derm_baseline_acc*100:.2f}% |  CBM acc: {new_acc_labels*100:.2f}%  |  fidelity error: {derm_fid_err*100:.2f}%")
print(f"HAM10000 - baseline acc: {ham_baseline_acc*100:.2f}%  |  CBM acc: {og_acc_labels*100:.2f}%  |  fidelity error: {ham_fid_err*100:.2f}%")

### Quantifying mapping

 Using Dice Similarity Coefficient (DSC) to quantify a visual mapping

In [ ]:
KEEP = 0.2

class ConceptClassTarget:
    """Routes GradCAM gradients through a specific concept head and class."""
    def __init__(self, concept_name, class_idx):
        self.concept_name = concept_name
        self.class_idx    = class_idx
    def __call__(self, model_output):
        if model_output.dim() == 1:
            model_output = model_output.unsqueeze(0)
        return model_output[:, self.class_idx]

class SingleConceptWrapper(nn.Module):
    """Wraps the multi-task model to output only one concept's logits.
    GradCAM expects a plain tensor output, not a dict."""
    def __init__(self, model, concept_name):
        super().__init__()
        self.model        = model
        self.concept_name = concept_name
    def forward(self, x):
        return self.model(x)[self.concept_name]

def dice_coefficient(heatmap1, heatmap2, threshold=0.5):
    if heatmap1.shape != heatmap2.shape:
        raise ValueError("Heatmaps must have the same shape.")

    mask1 = (heatmap1 >= threshold).astype(np.uint8)
    mask2 = (heatmap2 >= threshold).astype(np.uint8)

    intersection = np.sum(mask1 * mask2)
    sum_of_areas = np.sum(mask1) + np.sum(mask2)

    if sum_of_areas == 0:
        return 1.0  # Both masks are empty, consider them perfectly overlapping

    return (2. * intersection) / sum_of_areas

print("Dice coefficient function defined.")

def dice_topk(a, b, keep=KEEP):
    def mask(x):
        x = np.asarray(x, dtype=float)
        t = np.quantile(x, 1 - keep)
        return (x >= t) if t > x.min() else (x > t)
    A, B = mask(a), mask(b)
    denom = A.sum() + B.sum()
    return 0.0 if denom == 0 else 2.0 * (A & B).sum() / denom

def get_cam(method_name, model, target_layer):
    methods = {
        'gradcam':   GradCAM,
        'gradcam++': GradCAMPlusPlus,
        'scorecam':  ScoreCAM,
        'eigencam':  EigenCAM,
    }
    return methods[method_name.lower()](model=model, target_layers=[target_layer])

def suppress_cam_edges(cam, margin_frac=0.08, renormalise=True):
    h, w = cam.shape
    my, mx = int(h * margin_frac), int(w * margin_frac)
    mask = np.ones_like(cam)
    for i in range(my):
        mask[i, :] *= i / my
        mask[-i-1, :] *= i / my
    for j in range(mx):
        mask[:, j] *= j / mx
        mask[:, -j-1] *= j / mx
    cam_masked = cam * mask
    if renormalise and cam_masked.max() > 0:
        cam_masked = cam_masked / cam_masked.max()
    return cam_masked

def get_label_name(concept, class_idx, concepts_config):
    labels = concepts_config[concept]['labels']
    return labels[class_idx] if 0 <= class_idx < len(labels) else 'unknown'

def generate_concept_cams(model, image_tensor, concepts_config, cam_method_name, device):
    model.eval()
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        outputs = model(image_tensor)
    predicted_classes = {
        concept: logits.argmax(dim=1).item()
        for concept, logits in outputs.items()
    }
    concept_cams = {}

    for concept in concepts_config:
        pred_class = predicted_classes[concept]
        wrapper = SingleConceptWrapper(model, concept)
        wrapper.eval()
        target_layer = wrapper.model.concept_convs[concept]

        cam = get_cam(cam_method_name, wrapper, target_layer)
        grayscale_cam = cam(
            input_tensor=image_tensor,
            targets=[ConceptClassTarget(concept, pred_class)]
        )
        cam_suppressed = suppress_cam_edges(grayscale_cam[0], margin_frac=0.08, renormalise=True)
        concept_cams[concept] = cam_suppressed

    return concept_cams, predicted_classes

def load_ham_gradcam(ham_image_id):
    ham_img_path = Path(DATA_DIR) / f"{ham_image_id}.jpg"
    pil_image = Image.open(ham_img_path).convert('RGB')
    pil_image = transforms.Resize((224, 224))(pil_image)
    tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])(pil_image).unsqueeze(0)
    return pil_image, tensor

def get_ice_concept_maps(x, ham_model, explainer):
    if explainer.reducer is None:
        print("Warning: explainer.reducer is None. "
              "Train the reducer first with explainer.train_model(...).")
        return {}

    feature_maps = ham_model.get_feature(x, explainer.layer_name)
    h = explainer.reducer.transform(feature_maps)[0]  # shape: (H, W, n_components)

    concept_maps_dict = {
        f'ice_concept_{k}': h[:, :, k] for k in range(explainer.n_components)
    }
    return concept_maps_dict

def load_ham_ice(ham_image_id):
    ham_img_path = Path(DATA_DIR) / f"{ham_image_id}.jpg"
    pil_image = Image.open(ham_img_path).convert('RGB')
    pil_image = transforms.Resize((224, 224))(pil_image)
    image = transforms.ToTensor()(pil_image)
    image = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])(image).numpy()
    return np.transpose(image, (1, 2, 0))

In [ ]:
THRESHOLD = 1e-6
# MONO_CSV = os.path.join(HAM_OUTPUT_DIR, 'top_visual_mono.csv')
# POLY_CSV = os.path.join(HAM_OUTPUT_DIR, 'top_visual_poly.csv')
MONO_CSV = os.path.join(HAM_OUTPUT_DIR, 'top_visual_mono_topk.csv')
POLY_CSV = os.path.join(HAM_OUTPUT_DIR, 'top_visual_poly_topk.csv')
CSV_COLUMNS = ['image_id', 'label', 'grad_concept', 'ice_concept', 'dice']


def calculate_image_mappings(ham_id, derm_model, ham_model, Exp, cfg, device, threshold=THRESHOLD):
    """Return (mono_mapping, poly_mapping) for one image.
    Each is {grad_concept: (ice_concept, dice)}."""
    _, gradcam_tensor = load_ham_gradcam(ham_id)
    ice_array = load_ham_ice(ham_id)

    gradcam_cams, _ = generate_concept_cams(
        derm_model, gradcam_tensor, cfg['concepts'], cfg['cam_method'], device
    )
    ice_maps = get_ice_concept_maps(ice_array, ham_model, Exp)

    grad_names, ice_names = list(gradcam_cams), list(ice_maps)
    if not grad_names or not ice_names:
        return {}, {}

    dice = np.zeros((len(grad_names), len(ice_names)))
    for i, g in enumerate(grad_names):
        h, w = gradcam_cams[g].shape[:2]
        for j, ic in enumerate(ice_names):
            resized = cv2.resize(ice_maps[ic], dsize=(w, h), interpolation=cv2.INTER_LINEAR)
            dice[i, j] = dice_topk(gradcam_cams[g], resized)
    dice[dice < threshold] = 0.0

    # Polysemantic (greedy)
    poly = {}
    for i, g in enumerate(grad_names):
        j = int(dice[i].argmax())
        if dice[i, j] >= threshold:
            poly[g] = (ice_names[j], float(dice[i, j]))

    # Monosemantic (1:1, hungarian)
    rows, cols = linear_sum_assignment(dice, maximize=True)
    mono = {grad_names[r]: (ice_names[c], float(dice[r, c]))
            for r, c in zip(rows, cols) if dice[r, c] >= threshold}

    return mono, poly


def save_results(results, path):
    """Save {image_id: {...}} as a long-format CSV (one row per concept pair)."""
    rows = [(iid, r['label'], g, ice, d)
            for iid, r in results.items()
            for g, (ice, d) in r['mapping'].items()]
    pd.DataFrame(rows, columns=CSV_COLUMNS).to_csv(path, index=False)


def load_results(path):
    """Inverse of save_results. Preserves the saved (score-sorted) order."""
    df = pd.read_csv(path)
    results = {}
    for (iid, label), grp in df.groupby(['image_id', 'label'], sort=False):
        mapping = {g: (ice, d) for g, ice, d in
                   zip(grp['grad_concept'], grp['ice_concept'], grp['dice'])}
        results[iid] = {'label': label,
                        'mapping': mapping,
                        'total_dice': sum(d for _, d in mapping.values())}
    return results


def get_visual_mappings(meta, derm_model, ham_model, Exp, cfg, device):
    """Load mono/poly results from CSV if both exist, otherwise compute and save them.
    Returns (mono_results, poly_results), each sorted by total_dice descending:
        {image_id: {'label': str, 'mapping': {grad: (ice, dice)}, 'total_dice': float}}"""
    if os.path.exists(MONO_CSV) and os.path.exists(POLY_CSV):
        print(f"Loading cached results from {HAM_OUTPUT_DIR}")
        return load_results(MONO_CSV), load_results(POLY_CSV)

    mono_results, poly_results = {}, {}
    subset = meta[meta['dx'].isin(['mel', 'nv'])]
    for _, row in tqdm.tqdm(subset.iterrows(), total=len(subset), desc="Mapping mel/nv images"):
        mono, poly = calculate_image_mappings(row['image_id'], derm_model, ham_model, Exp, cfg, device)
        for results, mapping in ((mono_results, mono), (poly_results, poly)):
            results[row['image_id']] = {
                'label': row['dx'],
                'mapping': mapping,
                'total_dice': sum(d for _, d in mapping.values()),
            }

    def by_score(results):
        return dict(sorted(results.items(), key=lambda kv: kv[1]['total_dice'], reverse=True))

    mono_results, poly_results = by_score(mono_results), by_score(poly_results)
    save_results(mono_results, MONO_CSV)
    save_results(poly_results, POLY_CSV)
    print(f"Saved:\n  {MONO_CSV}\n  {POLY_CSV}")
    return mono_results, poly_results


mono_results, poly_results = get_visual_mappings(meta, derm_model, ham_model, Exp, cfg, device)

### Applying Grad-CAM to Derm7pt

In [ ]:
EPSILON = 1e-8

def ice_contour_overlay(original_pil_img, heatmap_data, threshold=0.5,
                         background=0.2, minmax=False, smooth=True,
                         contour_color=(255, 0, 0), thickness=3):
    w, h_img = original_pil_img.size
    h = np.array(heatmap_data, dtype=np.float32).copy()[None, ...]

    if minmax:
        h = h - h.min()

    h = h * (h > 0)
    for i in range(h.shape[0]):
        h[i] = h[i] / (h[i].max() + EPSILON)

    h = (h - threshold) * (1 / (1 - threshold))   # rescale so `threshold` -> 0

    if smooth:
        tsize = [1, h_img, w]
        h = sk_resize(h, tsize, order=1, mode='reflect', anti_aliasing=False)
    else:
        h = np.repeat(h, h_img // h.shape[1], axis=1)
        h = np.repeat(h, w // h.shape[2], axis=2)

    h = ((h > 0).astype("float") * (1 - background) + background)
    h = h[0]

    h_mask = np.repeat(h[..., None], 3, axis=-1)
    img_np = np.array(original_pil_img.convert("RGB")).astype(np.float32)
    out = img_np * h_mask

    h = h - h.min()
    h = h / (h.max() + EPSILON)

    out = out.astype(np.uint8)
    mask_uint8 = (h > 0.5).astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, contours, -1, contour_color, thickness)

    return Image.fromarray(out)

def visualise_all_concepts(image, concept_cams, predicted_classes, concepts_config):
    overlayed_images_with_titles = {}
    for concept, cam in concept_cams.items():
        pred_idx = predicted_classes[concept]
        pred_name = get_label_name(concept, pred_idx, concepts_config)
        concept_display = concept.replace('_', ' ').title()
        title = f'{concept_display} [predicted={pred_name}]'
        overlayed_images_with_titles[title] = ice_contour_overlay(image, cam)
    return overlayed_images_with_titles

### Extracting HAM10000 Heatmaps

In [ ]:
def get_all_overlayed_concept_images(original_pil_img, ham_model, explainer, x):
    ice_maps = get_ice_concept_maps(x, ham_model, explainer)

    overlayed_images_with_titles = {}
    for i, (name, cam) in enumerate(ice_maps.items()):
        title = f'ICE Concept {i}'
        overlayed_images_with_titles[title] = ice_contour_overlay(original_pil_img, cam)

    return overlayed_images_with_titles

def plot_image_grid(image_dict, suptitle=None, ncols=5):
    if not image_dict:
        print("No images to display.")
        return

    n = len(image_dict)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, (title, img) in zip(axes, image_dict.items()):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.01)

    plt.tight_layout()
    from IPython.display import display
    display(fig)
    plt.close()

In [ ]:
def plot_paired_heatmaps(original_pil_img, gradcam_cams, ice_maps, mapping_data, mapping_type, concepts_config, image_id, predicted_classes):
    """
    Plots paired GradCAM and ICE heatmaps based on mapping data.

    Args:
        original_pil_img (PIL.Image.Image): The original image.
        gradcam_cams (dict): Dictionary of all generated GradCAM heatmaps.
        ice_maps (dict): Dictionary of all generated ICE heatmaps.
        mapping_data (list or dict): The mapping data (Hungarian list of tuples or Polysemantic dict).
        mapping_type (str): 'hungarian' or 'polysemantic'.
        concepts_config (dict): Configuration for concepts.
        image_id (str): The ID of the image being processed.
        predicted_classes (dict): Predicted classes for GradCAM concepts.
    """
    print(f"\n--- Paired {mapping_type.capitalize()} Mappings for {image_id} ---")
    for grad_concept, ice_concept, dice_score in mapping_data:
        display_images = {}

        # GradCAM visualization
        pred_idx_gradcam = predicted_classes.get(grad_concept, -1)
        pred_name_gradcam = get_label_name(grad_concept, pred_idx_gradcam, concepts_config)
        gradcam_display = grad_concept.replace('_', ' ').title()
        title_gradcam = f'GradCAM {gradcam_display} ({pred_name_gradcam})'
        display_images[title_gradcam] = ice_contour_overlay(original_pil_img, gradcam_cams[grad_concept])

        # ICE visualization
        ice_concept_idx = int(ice_concept.split('_')[-1])
        title_ice = f'ICE Concept {ice_concept_idx}'
        display_images[title_ice] = ice_contour_overlay(original_pil_img, ice_maps[ice_concept])

        plot_image_grid(display_images, suptitle=f'Dice Score: {dice_score:.4f}', ncols=2)

In [ ]:
from skimage.transform import resize as sk_resize
N_PER_CLASS = 8

RESULTS = {'Hungarian': mono_results, 'Polysemantic': poly_results}

for mapping_name, results in RESULTS.items():
    print(f"\n--- Displaying paired heatmaps for Top {mapping_name} Mappings ---")
    counts = {'mel': 0, 'nv': 0}
    selected = []
    for ham_id, result in results.items():
        label = result['label']
        if label in counts and counts[label] < N_PER_CLASS:
            selected.append((ham_id, result))
            counts[label] += 1
        if all(c >= N_PER_CLASS for c in counts.values()):
            break

    for ham_id, result in selected:
        print(f"\n-- HAM10000 image ID: {ham_id}  [Diagnosis: {result['label']}] --")
        print(f"   Total {mapping_name} Dice Score: {result['total_dice']:.4f}")

        pil_image_gradcam, ham_image_tensor_gradcam = load_ham_gradcam(ham_id)
        ham_image_array_ice = load_ham_ice(ham_id)

        plot_image_grid({'Original Image': pil_image_gradcam},
                        suptitle=f'Original Image for {ham_id}', ncols=1)

        gradcam_cams, predicted_classes = generate_concept_cams(
            derm_model, ham_image_tensor_gradcam, cfg['concepts'], cfg['cam_method'], device
        )
        ice_maps = get_ice_concept_maps(ham_image_array_ice, ham_model, Exp)
        mapping_data = [(g, ice, d) for g, (ice, d) in result['mapping'].items()]

        if mapping_data:
            plot_paired_heatmaps(
                pil_image_gradcam,
                gradcam_cams,
                ice_maps,
                mapping_data,
                mapping_name.lower(),
                cfg['concepts'],
                ham_id,
                predicted_classes,
            )
        else:
            print(f"No valid {mapping_name} mapping data to display for {ham_id}.")
        print("\n")

    print(f"Count of 'melanoma' images displayed: {counts['mel']}")
    print(f"Count of 'nevus' images displayed: {counts['nv']}")

### Visualising results

In [ ]:
KEEP = 0.2
N_RUNS = 5
SEEDS = [0, 1, 2, 3, 4]

DERM_MODEL_CHECKPOINTS = [
    os.path.join(OUTPUT_DIR, f'best_model_run{i+1}.pth') for i in range(N_RUNS)
]

def csv_paths(tag):
    return (os.path.join(HAM_OUTPUT_DIR, f'top_visual_mono_{tag}_topk.csv'),
            os.path.join(HAM_OUTPUT_DIR, f'top_visual_poly_{tag}_topk.csv'))


def _mask(x, keep=KEEP):
    """Same top-k rule as dice_topk, returned as a flat boolean vector."""
    x = np.asarray(x, dtype=float)
    t = np.quantile(x, 1 - keep)
    return ((x >= t) if t > x.min() else (x > t)).ravel()


def dice_from_masks(A, B):
    """A: (n_grad, n_pixels), B: (n_ice, n_pixels) boolean -> (n_grad, n_ice) Dice."""
    A, B = A.astype(np.float32), B.astype(np.float32)
    inter = A @ B.T
    denom = A.sum(1)[:, None] + B.sum(1)[None, :]
    return np.divide(2 * inter, denom, out=np.zeros_like(inter), where=denom > 0)


def assign(dice, grad_names, ice_names, threshold=THRESHOLD):
    dice = np.where(dice < threshold, 0.0, dice)
    poly = {}
    for i, g in enumerate(grad_names):
        j = int(dice[i].argmax())
        if dice[i, j] >= threshold:
            poly[g] = (ice_names[j], float(dice[i, j]))
    rows, cols = linear_sum_assignment(dice, maximize=True)
    mono = {grad_names[r]: (ice_names[c], float(dice[r, c]))
            for r, c in zip(rows, cols) if dice[r, c] >= threshold}
    return mono, poly


def get_all_runs(meta, derm_models, ham_model, Exp, cfg, device, keep=KEEP, suffix=''):
    """derm_models = {'run0': model, ...}. Runs whose CSVs exist are loaded; the rest are
    computed together in ONE pass over the images (ICE maps shared across runs).
    Returns {run_id: {'mono': results, 'poly': results}}."""
    paths = {r: csv_paths(r + suffix) for r in derm_models}
    todo = [r for r, (m, p) in paths.items() if not (os.path.exists(m) and os.path.exists(p))]
    out = {r: {'mono': {}, 'poly': {}} for r in todo}

    if todo:
        subset = meta[meta['dx'].isin(['mel', 'nv'])]
        for _, row in tqdm.tqdm(subset.iterrows(), total=len(subset),
                                desc=f'Mapping {len(todo)} runs'):
            ham_id = row['image_id']
            _, gradcam_tensor = load_ham_gradcam(ham_id)
            ice_maps = get_ice_concept_maps(load_ham_ice(ham_id), ham_model, Exp)  # once per image
            ice_names = list(ice_maps)
            ice_masks = None

            for r in todo:
                cams, _ = generate_concept_cams(derm_models[r], gradcam_tensor,
                                                cfg['concepts'], cfg['cam_method'], device)
                grad_names = list(cams)
                if not grad_names or not ice_names:
                    mono, poly = {}, {}
                else:
                    if ice_masks is None:   # resize + mask the ICE maps once, reuse for every run
                        h, w = cams[grad_names[0]].shape[:2]
                        ice_masks = np.stack([
                            _mask(cv2.resize(ice_maps[n], dsize=(w, h),
                                             interpolation=cv2.INTER_LINEAR), keep)
                            for n in ice_names])
                    A = np.stack([_mask(cams[g], keep) for g in grad_names])
                    mono, poly = assign(dice_from_masks(A, ice_masks), grad_names, ice_names)

                for kind, mapping in (('mono', mono), ('poly', poly)):
                    out[r][kind][ham_id] = {'label': row['dx'], 'mapping': mapping,
                                            'total_dice': sum(d for _, d in mapping.values())}

        for r in todo:
            for kind, path in zip(('mono', 'poly'), paths[r]):
                out[r][kind] = dict(sorted(out[r][kind].items(),
                                           key=lambda kv: kv[1]['total_dice'], reverse=True))
                save_results(out[r][kind], path)
            print(f'Saved {r}: {paths[r][0]}')

    runs = {}
    for r, (mono_csv, poly_csv) in paths.items():
        runs[r] = out[r] if r in todo else {'mono': load_results(mono_csv),
                                            'poly': load_results(poly_csv)}
    return runs

derm_models = {}
for i, checkpoint_path in enumerate(DERM_MODEL_CHECKPOINTS):
    m = Derm7ptMultiTaskModel(cfg['concepts']).to(device)
    m.load_state_dict(torch.load(checkpoint_path, map_location=device))
    m.eval()
    derm_models[f'run{i}'] = m

runs = get_all_runs(meta, derm_models, ham_model, Exp, cfg, device)


In [ ]:
def results_to_df(results):
    rows = [(iid, r['label'], g, ice, d)
            for iid, r in results.items()
            for g, (ice, d) in r['mapping'].items()]
    return pd.DataFrame(rows, columns=['image_id', 'label', 'grad_concept', 'ice_concept', 'dice'])


def runs_to_df(runs, kind):
    frames = []
    for run_id, res in runs.items():
        d = results_to_df(res[kind])
        d['run'] = run_id
        frames.append(d)
    return pd.concat(frames, ignore_index=True)


# Confusion matrices
def _share(df, cls, grads, ices):
    sub = df[df['label'] == cls]
    ct = pd.crosstab(sub['grad_concept'], sub['ice_concept'], normalize='index')
    return ct.reindex(index=grads, columns=ices, fill_value=0.0)


def plot_class_matrices(df, title):
    grads = sorted(df['grad_concept'].unique())
    ices = sorted(df['ice_concept'].unique())
    mel, nv = _share(df, 'mel', grads, ices), _share(df, 'nv', grads, ices)

    absolute_top = max(mel.values.max(), nv.values.max())
    print(f'absolute top: {absolute_top:.4f}')

    diff = mel - nv
    lim = np.abs(diff.values).max() or 1.0
    panels = [(mel, 'Melanoma', 'viridis', 0, absolute_top),
              (nv, 'Nevus', 'viridis', 0, absolute_top),
              (diff, 'Melanoma - Nevus', 'RdBu_r', -lim, lim)]

    fig, axes = plt.subplots(1, 3, figsize=(19, 5))
    for k, (ax, (m, name, cmap, lo, hi)) in enumerate(zip(axes, panels)):
        im = ax.imshow(m.values, cmap=cmap, vmin=lo, vmax=hi, aspect='auto')
        ax.set_xticks(range(len(ices)))
        ax.set_xticklabels([str(i) for i in ices], rotation=90)
        ax.set_yticks(range(len(grads)))
        ax.set_yticklabels(grads if k == 0 else [])
        ax.set_title(name)
        for i in range(m.shape[0]):
            for j in range(m.shape[1]):
                ax.text(j, i, f'{m.iloc[i, j]:.2f}', ha='center', va='center',
                         fontsize=6, color='black')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xlabel('ICE concept')
        fig.colorbar(im, ax=ax, fraction=0.046, label='share of images')
    axes[0].set_ylabel('GradCAM concept')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# Top concept (also saves % for later)
def get_top_concepts(df, kind_name, enforce_one_to_one=False):
    print(f'\n--- {kind_name} Top Concepts ---')
    mappings_out, pcts_out = {}, {}
    for label, l_grp in df.groupby('label'):
        print(f'\nDiagnosis: {label.upper()}')

        ct = pd.crosstab(l_grp['grad_concept'], l_grp['ice_concept'])
        grads = sorted(l_grp['grad_concept'].unique())
        ices = sorted(l_grp['ice_concept'].unique())
        ct = ct.reindex(index=grads, columns=ices, fill_value=0)

        if enforce_one_to_one:
            row_ind, col_ind = linear_sum_assignment(ct.values, maximize=True)
            mappings = {grads[r]: ices[c] for r, c in zip(row_ind, col_ind)}
        else:
            mappings = {g: ct.loc[g].idxmax() for g in grads}

        pcts = {}
        for g in grads:
            assigned = mappings.get(g)
            total = ct.loc[g].sum()
            top_count = ct.loc[g, assigned] if assigned in ct.columns else 0
            pcts[g] = (top_count / total) * 100 if total > 0 else 0.0
            print(f'{g:25s} {assigned}')

        mappings_out[label] = mappings
        pcts_out[label] = pcts
    return mappings_out, pcts_out

def run_stats_for_mapping(df, mappings):
    rows = []
    for (run, cls), grp in df.groupby(['run', 'label']):
        for g, gg in grp.groupby('grad_concept'):
            top = mappings[cls].get(g)
            hit = gg[gg['ice_concept'] == top]
            rows.append({'run': run, 'label': cls, 'grad_concept': g, 'top_ice': top,
                         'strength': hit['dice'].mean() if len(hit) else np.nan,
                         'strength_all': gg['dice'].mean(),
                         'consistency': (gg['ice_concept'] == top).mean()})
    return pd.DataFrame(rows)


def plot_run_bar(stats, metric, title, ylabel, err_metric=None, chance=None):
    grads = sorted(stats['grad_concept'].unique())
    mean = stats.groupby(['grad_concept', 'label'])[metric].mean()
    if err_metric is not None:
        err = stats.groupby(['grad_concept', 'label'])[err_metric].std()
    x, w = np.arange(len(grads)), 0.35

    fig, ax = plt.subplots(figsize=(12, 5))
    for k, (cls, name) in enumerate([('mel', 'Melanoma'), ('nv', 'Nevus')]):
        m = mean.xs(cls, level='label').reindex(grads)
        yerr = err.xs(cls, level='label').reindex(grads).fillna(0) if err_metric is not None else None
        ax.bar(x + (k - 0.5) * w, m, w, yerr=yerr, capsize=3 if yerr is not None else 0, label=name, color='#d62728' if cls == 'mel' else '#1f77b4')
    if chance is not None:
        ax.axhline(chance, color='gray', linestyle='--', linewidth=1, label='Chance level (10%)')
    ax.set_xticks(x)
    ax.set_xticklabels(grads, rotation=45, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


saved_mappings = {}        # saved_mappings[kind][label][grad] = top ICE concept
saved_match_pcts = {}      # saved_match_pcts[kind][label][grad] = match %

for kind, name in (('mono', 'Hungarian (monosemantic)'), ('poly', 'Polysemantic')):
    df = runs_to_df(runs, kind)
    df['ice_concept'] = df['ice_concept'].replace({'ice_concept_1': 'ice_concept_5',
                                                   'ice_concept_5': 'ice_concept_1'})

    plot_class_matrices(df, f'{name}: share of images assigning each ICE concept')

    mappings, pcts = get_top_concepts(df, name, enforce_one_to_one=(kind == 'mono'))
    saved_mappings[kind] = mappings
    saved_match_pcts[kind] = pcts

    stats = run_stats_for_mapping(df, mappings)
    plot_run_bar(stats, 'strength',
                 f'{name}: Strength of best ICE-concept match, per derm concept', 'Best-match cosine similarity (mean ± std)',
                 err_metric='strength_all')

    plot_run_bar(stats, 'consistency',
                 f'{name}: Consistency of best-match ICE concept across images',
                 '% of images where this ICE concept was the top match',
                 chance=0.10)